# Chronos 2 — Wind
Zero-shot 24-hour rolling forecasts, with and without NWP covariates.

This notebook reports the overall MAE, RMSE, and aQL result.


In [ ]:
# !pip install -r requirements/chronos2.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "requirements").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "wind"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from preprocessing import load_wind, describe_wind, NWP_FEATURES, TEST_START, SEED, CONTEXT_LENGTH, PREDICTION_LENGTH, SITES
from src.metrics import print_metrics
from power_conversion import convert_wind_result_to_power

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=[q for q in OUTPUT_QUANTILES if q not in (0.025,0.975)]
OUTPUT_DIR=ROOT/"outputs"/"predictions"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data = load_wind(ROOT / "data" / "wind")
print(describe_wind(data))

from tqdm.auto import tqdm
from chronos import BaseChronosPipeline


In [ ]:
VARIANT = "with_covariates"  # or "without_covariates"
DEVICE = "cuda"
pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)


In [ ]:
rolling = {s: data.context_by_site[s].copy() for s in SITES}
rows = []
for i in tqdm(range(len(data.daily_by_site[SITES[0]]))):
    day = {s: data.daily_by_site[s][i][1].copy() for s in SITES}
    for site in SITES:
        context = rolling[site].sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
        kwargs = dict(prediction_length=PREDICTION_LENGTH, quantile_levels=MODEL_QUANTILES, id_column="item_id", timestamp_column="timestamp", target="target")
        if VARIANT == "with_covariates":
            pred = pipeline.predict_df(context, future_df=day[site].drop(columns="target"), **kwargs)
        else:
            pred = pipeline.predict_df(context[["item_id","timestamp","target"]], **kwargs)
        rows.append(pred)
    for site in SITES:
        rolling[site] = pd.concat([rolling[site], day[site]], ignore_index=True)

result = pd.concat(rows, ignore_index=True)
if "predictions" in result.columns: result = result.rename(columns={"predictions":"prediction"})
if "prediction" not in result.columns: result["prediction"] = result["0.5"]
result = result.merge(data.test[["item_id","timestamp","target"]], on=["item_id","timestamp"], how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR / f"chronos2_{VARIANT}.csv", index=False)
print_metrics(result, QUANTILES, label=f"chronos2_{VARIANT}")

# Wind speed -> wind power evaluation
power_result=convert_wind_result_to_power(result)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in power_result.columns:
        power_result[str(_q)] = np.nan
power_result.to_csv(OUTPUT_DIR/f"chronos2_{VARIANT}_power.csv",index=False)
print_metrics(power_result,QUANTILES,label=f"chronos2_{VARIANT} [wind power]")
